## 从 Wtie 成果构建时间域低频模型

这个 notebook 读取已经保存好的 7 口井井震标定成果（AI + 时深表），构建 `LfmTimeWell` 输入，调用 `lfm_time.py` 生成时间域低频模型，并做三维与剖面检查。


In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

try:
    import pyvista as pv
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pyvista is required for the visualization cells in this notebook. "
        "Please install it in the notebook kernel environment first."
    ) from exc

from cup.petrel.load import (
    import_checkshots_petrel,
    import_interpretation_petrel,
    import_well_heads_petrel,
)
from cup.seismic.lfm_time import LfmTimeWell, build_lfm_time_model
from cup.seismic.process import TargetLayer, interpolate_interpretation_surface
from cup.seismic.survey import open_survey
from wtie.processing import grid

pv.set_jupyter_backend("static")

In [ ]:
data_root = repo_root / "data"
# debug_dir = data_root / "debug"
wtie_root = data_root / "vertical_well_wtie_output"
ai_dir = wtie_root / "ai"
tdtable_dir = wtie_root / "tdtable"
wavelet_dir = wtie_root / "wavelet"  # 当前 notebook 不参与低频模型构建，仅保留路径

well_heads_file = data_root / "raw" / "well_heads"

seismic_file = data_root / "raw" / "mero se 0116_1ms_new_84_coord.Sgy"
segy_iline = 5
segy_xline = 21
segy_istep = 1
segy_xstep = 4

horizon_files = {
    "bve_top": data_root / "interpre_time" / "bve_top_t",
    "bve_bot": data_root / "interpre_time" / "bve_bot_t",
    "itp_bot": data_root / "interpre_time" / "itp_bot_t",
}

for path in [ai_dir, tdtable_dir, well_heads_file, seismic_file, *horizon_files.values()]:
    if not path.exists():
        raise FileNotFoundError(path)

wtie_root

In [ ]:
STEM_PATTERN = re.compile(r"^(?P<well_name>.+)_(?P<offset>-?\d+(?:\.\d+)?)$")


def parse_artifact_stem(stem: str) -> tuple[str, float]:
    match = STEM_PATTERN.match(stem)
    if match is None:
        raise ValueError(f"Unrecognized artifact stem: {stem}")
    return match.group("well_name"), float(match.group("offset"))


def collect_wtie_artifacts(ai_dir: Path, tdtable_dir: Path) -> pd.DataFrame:
    ai_map = {path.stem: path for path in sorted(ai_dir.glob("*.csv"))}
    tdt_map = {path.stem: path for path in sorted(tdtable_dir.glob("*.txt"))}

    common_stems = sorted(set(ai_map) & set(tdt_map))
    if not common_stems:
        raise ValueError("No paired AI/tdtable artifacts were found.")

    missing_ai = sorted(set(tdt_map) - set(ai_map))
    missing_tdt = sorted(set(ai_map) - set(tdt_map))
    if missing_ai:
        print("Stems missing AI:", missing_ai)
    if missing_tdt:
        print("Stems missing tdtable:", missing_tdt)

    records = []
    for stem in common_stems:
        well_name, interpretation_offset = parse_artifact_stem(stem)
        records.append(
            {
                "stem": stem,
                "well_name": well_name,
                "interpretation_offset": interpretation_offset,
                "ai_file": ai_map[stem],
                "tdtable_file": tdt_map[stem],
            }
        )

    return pd.DataFrame.from_records(records).sort_values(["well_name", "interpretation_offset"]).reset_index(drop=True)


def load_ai_log_from_csv(csv_file: Path) -> grid.Log:
    df = pd.read_csv(csv_file)
    required_cols = {"twt_s", "ai"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"AI CSV is missing columns {sorted(missing)}: {csv_file}")

    twt = df["twt_s"].to_numpy(dtype=float)
    ai = df["ai"].to_numpy(dtype=float)
    order = np.argsort(twt)
    twt = twt[order]
    ai = ai[order]
    return grid.Log(ai, twt, "twt", name="AI", allow_nan=False)


def build_target_layer(geometry: dict) -> TargetLayer:
    interpolated = {}
    for horizon_name, horizon_file in horizon_files.items():
        raw_df = import_interpretation_petrel(horizon_file)
        interp_df = interpolate_interpretation_surface(
            interpretation_df=raw_df,
            geometry=geometry,
            outlier_threshold=0.02,
            min_neighbor_count=2,
            keep_nan=True,
        )
        interpolated[horizon_name] = interp_df

    return TargetLayer(
        interpolated_horizon_dfs=interpolated,
        geometry=geometry,
        horizon_names=list(horizon_files.keys()),
        # debug_dir=debug_dir,
    )


def build_lfm_wells(artifact_df: pd.DataFrame, well_heads_df: pd.DataFrame) -> list[LfmTimeWell]:
    head_lookup = well_heads_df.copy()
    head_lookup["_name_norm"] = head_lookup["Name"].astype(str).str.strip()
    head_lookup = head_lookup.set_index("_name_norm", drop=False)

    wells = []
    for row in artifact_df.itertuples(index=False):
        if row.well_name not in head_lookup.index:
            raise KeyError(f"Well '{row.well_name}' not found in well heads file.")

        head = head_lookup.loc[row.well_name]
        ai_log = load_ai_log_from_csv(row.ai_file)
        tdtable = import_checkshots_petrel(row.tdtable_file, depth_domain="tvdss")

        wells.append(
            LfmTimeWell(
                well_name=row.well_name,
                property_name="AI",
                property_log=ai_log,
                time_depth_table=tdtable,
                x=float(head["Surface X"]),
                y=float(head["Surface Y"]),
                metadata={
                    "artifact_stem": row.stem,
                    "interpretation_offset": float(row.interpretation_offset),
                    "kb": float(head["Well datum value"]),
                },
            )
        )

    return wells


def make_pyvista_grid(result, scalar_name: str = "AI"):
    x_spacing = float(result.xlines[1] - result.xlines[0]) if result.xlines.size > 1 else 1.0
    y_spacing = float(result.ilines[1] - result.ilines[0]) if result.ilines.size > 1 else 1.0
    z_spacing = float(result.samples[1] - result.samples[0]) if result.samples.size > 1 else 1.0

    image = pv.ImageData(
        dimensions=(int(result.xlines.size), int(result.ilines.size), int(result.samples.size)),
        spacing=(x_spacing, y_spacing, z_spacing),
        origin=(float(result.xlines[0]), float(result.ilines[0]), float(result.samples[0])),
    )
    image.point_data[scalar_name] = np.transpose(result.volume, (1, 0, 2)).ravel(order="F")
    image.point_data["variance"] = np.transpose(result.variance_volume, (1, 0, 2)).ravel(order="F")
    image.set_active_scalars(scalar_name)
    return image


def make_well_location_polydata(wells: list[LfmTimeWell], survey, z_value: float):
    points = []
    labels = []
    for well in wells:
        if well.x is not None and well.y is not None:
            iline, xline = survey.coord_to_line(float(well.x), float(well.y))
        elif well.inline is not None and well.xline is not None:
            iline, xline = float(well.inline), float(well.xline)
        else:
            continue

        points.append([float(xline), float(iline), float(z_value)])
        labels.append(well.well_name)

    if points:
        point_array = np.asarray(points, dtype=float)
    else:
        point_array = np.empty((0, 3), dtype=float)
    poly = pv.PolyData(point_array)
    return poly, labels

In [ ]:
artifact_df = collect_wtie_artifacts(ai_dir=ai_dir, tdtable_dir=tdtable_dir)
well_heads_df = import_well_heads_petrel(well_heads_file)

print(f"Found {len(artifact_df)} paired wells.")
artifact_df

In [ ]:
survey = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": segy_iline,
        "xline": segy_xline,
        "istep": segy_istep,
        "xstep": segy_xstep,
    },
)
geometry = survey.query_geometry(domain="time")
target_layer = build_target_layer(geometry)

print("Horizon names:", target_layer.horizon_names)
print("Zones:", target_layer.iter_zones())
geometry

In [ ]:
lfm_wells = build_lfm_wells(artifact_df, well_heads_df)

well_summary = pd.DataFrame(
    [
        {
            "well_name": well.well_name,
            "x": well.x,
            "y": well.y,
            "ai_samples": int(well.property_log.size),
            "twt_min": float(well.property_log.basis.min()),
            "twt_max": float(well.property_log.basis.max()),
            "tdt_samples": int(well.time_depth_table.size),
            "interpretation_offset": well.metadata.get("interpretation_offset"),
        }
        for well in lfm_wells
    ]
)
well_summary

In [ ]:
lfm_result = build_lfm_time_model(
    target_layer=target_layer,
    wells=lfm_wells,
    survey=survey,
    n_slices=20,
    variogram="spherical",
    exact=True,
    nugget=0.0,
)

print("volume shape:", lfm_result.volume.shape)
print("variance shape:", lfm_result.variance_volume.shape)
print("property:", lfm_result.metadata["property_name"])
print("zones:", lfm_result.metadata["zone_names"])
print("wells:", lfm_result.metadata["well_names"])

In [ ]:
ilines = lfm_result.ilines
xlines = lfm_result.xlines
samples = lfm_result.samples
volume = lfm_result.volume

i_il = len(ilines) // 2
i_xl = len(xlines) // 2
i_t = len(samples) // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

im0 = axes[0].imshow(
    volume[i_il, :, :].T,
    aspect="auto",
    origin="upper",
    extent=[xlines[0], xlines[-1], samples[-1], samples[0]],
    cmap="viridis",
)
axes[0].set_title(f"Inline section @ {ilines[i_il]:.0f}")
axes[0].set_xlabel("Xline")
axes[0].set_ylabel("TWT (s)")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(
    volume[:, i_xl, :].T,
    aspect="auto",
    origin="upper",
    extent=[ilines[0], ilines[-1], samples[-1], samples[0]],
    cmap="viridis",
)
axes[1].set_title(f"Xline section @ {xlines[i_xl]:.0f}")
axes[1].set_xlabel("Inline")
axes[1].set_ylabel("TWT (s)")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

im2 = axes[2].imshow(
    volume[:, :, i_t],
    aspect="auto",
    origin="lower",
    extent=[xlines[0], xlines[-1], ilines[0], ilines[-1]],
    cmap="viridis",
)
axes[2].set_title(f"Time slice @ {samples[i_t]:.3f} s")
axes[2].set_xlabel("Xline")
axes[2].set_ylabel("Inline")
fig.colorbar(im2, ax=axes[2], shrink=0.85)

plt.show()

In [ ]:
coverage_rows = []
for zone_name, zone_stats in lfm_result.coverage_stats["zones"].items():
    coverage_rows.append(
        {
            "zone": zone_name,
            "n_slices": len(zone_stats["slice_control_counts"]),
            "min_controls": int(np.min(zone_stats["slice_control_counts"])),
            "max_controls": int(np.max(zone_stats["slice_control_counts"])),
            "mean_controls": float(np.mean(zone_stats["slice_control_counts"])),
        }
    )

pd.DataFrame(coverage_rows)